# LeetCode 3277 — Maximum XOR Score Subarray Queries bằng Python

- Link: https://leetcode.com/problems/maximum-xor-score-subarray-queries/
- Độ khó: **Hard**
- Chủ đề: **DP khoảng** `score[i][j] = score[i][j−1] ⊕ score[i+1][j]` (giống tam giác Pascal mod 2), **DP max theo khoảng** `best[i][j]` trả lời mỗi truy vấn `O(1)`; liên hệ **định lý Lucas** (`C(m, k)` lẻ ⇔ `k ⊆ m`)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumSubarrayXor(self, nums: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Hiểu "XOR score": lặp phép `a[i] ← a[i] ⊕ a[i+1]` rồi bỏ phần tử cuối tới khi còn một số — đây là tam giác Pascal **modulo 2**.
2. Rút ra truy hồi khoảng: `score(i, j) = score(i, j−1) ⊕ score(i+1, j)`.
3. `best(i, j)` = max score của mọi đoạn con trong `[i, j]` `= max(score(i, j), best(i, j−1), best(i+1, j))` ⇒ bảng `O(n²)`, truy vấn `O(1)`.
4. Công thức đóng (Lucas): `score(a) = ⊕_{k : k & (m−1) = k} a[k]` với `m = len(a)`.
5. Kiểm chứng bằng mô phỏng.

## 2. Đề bài & ví dụ

Cho `nums` và các truy vấn `[l, r]`. Với mỗi truy vấn, tìm **XOR score lớn nhất** của một đoạn con của `nums[l..r]`. XOR score của mảng `a`: lặp lại "đồng thời thay `a[i]` bằng `a[i] ⊕ a[i+1]` với mọi `i` trừ chỉ số cuối, rồi xóa phần tử cuối" tới khi còn một phần tử.

**Ví dụ 1:** `nums = [2,8,4,32,16,1]`, `queries = [[0,2],[1,4],[0,5]]` → `[12,60,60]`.
**Ví dụ 2:** `nums = [0,7,3,2,8,5,1]`, `queries = [[0,3],[1,5],[2,4],[2,6],[5,6]]` → `[7,14,11,14,5]`.

**Ràng buộc:** `1 ≤ n ≤ 2000`; `0 ≤ nums[i] ≤ 2^31 − 1`; `1 ≤ q ≤ 10^5`.

In [1]:
from typing import List

examples = [([2, 8, 4, 32, 16, 1], [[0, 2], [1, 4], [0, 5]], [12, 60, 60]),
            ([0, 7, 3, 2, 8, 5, 1], [[0, 3], [1, 5], [2, 4], [2, 6], [5, 6]], [7, 14, 11, 14, 5])]
for nums, qs, exp in examples:
    print(nums, qs, '-> kỳ vọng', exp)

[2, 8, 4, 32, 16, 1] [[0, 2], [1, 4], [0, 5]] -> kỳ vọng [12, 60, 60]
[0, 7, 3, 2, 8, 5, 1] [[0, 3], [1, 5], [2, 4], [2, 6], [5, 6]] -> kỳ vọng [7, 14, 11, 14, 5]


## 3. Kiến thức nền

### 3.1. Mô phỏng định nghĩa

In [2]:
def xor_score(a):
    a = list(a)
    while len(a) > 1:
        a = [a[i] ^ a[i + 1] for i in range(len(a) - 1)]
    return a[0]

print('score [2, 8, 4] =', xor_score([2, 8, 4]), '| score [8, 4] =', xor_score([8, 4]))

score [2, 8, 4] = 6 | score [8, 4] = 12


### 3.2. Truy hồi khoảng

Sau bước đầu, mảng `[a_i ⊕ a_{i+1}, …]`. Quan sát: score của `a[i..j]` = score của hàng "sau một bước", mà hàng đó lại là XOR của hai mảng con dịch nhau. Do tính tuyến tính của XOR: `score(i, j) = score(i, j−1) ⊕ score(i+1, j)` (giống `C(m, k) = C(m−1, k−1) + C(m−1, k)` mod 2).

In [3]:
import random
r_ = random.Random(1)
for _ in range(200):
    a = [r_.randint(0, 100) for _ in range(r_.randint(2, 9))]
    assert xor_score(a) == xor_score(a[:-1]) ^ xor_score(a[1:])
print('score(i, j) = score(i, j-1) xor score(i+1, j): đúng trên 200 mảng ngẫu nhiên.')

score(i, j) = score(i, j-1) xor score(i+1, j): đúng trên 200 mảng ngẫu nhiên.


### 3.3. Công thức Lucas

Phần tử `a[k]` xuất hiện trong score với hệ số `C(m−1, k) mod 2`; theo định lý Lucas, `C(m−1, k)` lẻ ⇔ `k & (m−1) == k`.

In [4]:
def lucas_score(a):
    m = len(a)
    s = 0
    for k, v in enumerate(a):
        if k & (m - 1) == k:
            s ^= v
    return s

for _ in range(200):
    a = [r_.randint(0, 100) for _ in range(r_.randint(1, 12))]
    assert lucas_score(a) == xor_score(a)
print('Công thức Lucas khớp mô phỏng. Chỉ số được dùng khi m = 4:', [k for k in range(4) if k & 3 == k], '| khi m = 5:', [k for k in range(5) if k & 4 == k])

Công thức Lucas khớp mô phỏng. Chỉ số được dùng khi m = 4: [0, 1, 2, 3] | khi m = 5: [0, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1, truy vấn `[0, 2]` trên `[2, 8, 4]`:

| đoạn | score |
|---|---|
| `[2]`, `[8]`, `[4]` | 2, 8, 4 |
| `[2,8]` | 10 |
| `[8,4]` | 12 |
| `[2,8,4]` | `10 ⊕ 12 = 6` |

Max `12` ✓. Bảng `best[i][j]` gộp: `best[0][2] = max(6, best[0][1] = 10, best[1][2] = 12) = 12`.

## 5. Cách 1 — Brute Force: mô phỏng mọi đoạn con của mỗi truy vấn

In [5]:
def brute(nums, queries):
    ans = []
    for l, r in queries:
        ans.append(max(xor_score(nums[i:j + 1]) for i in range(l, r + 1) for j in range(i, r + 1)))
    return ans

for nums, qs, exp in examples:
    print(brute(nums, qs), '| kỳ vọng', exp)

[12, 60, 60] | kỳ vọng [12, 60, 60]
[7, 14, 11, 14, 5] | kỳ vọng [7, 14, 11, 14, 5]


**Độ phức tạp:** `O(q · n² · n²)` — chỉ cho dữ liệu rất nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

```
for i = n−1 .. 0:
    score[i][i] = best[i][i] = nums[i]
    for j = i+1 .. n−1:
        score[i][j] = score[i][j−1] ^ score[i+1][j]
        best[i][j]  = max(score[i][j], best[i][j−1], best[i+1][j])
answer(l, r) = best[l][r]
```

Chỉ cần **hàng `i + 1`** khi tính hàng `i` cho `score`, nhưng `best` phải giữ cả bảng (truy vấn bất kỳ) — `≈ 2·10^6` ô với `n = 2000`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Lucas cho từng đoạn + max theo từng truy vấn (`O(n³)` tiền xử lý)

In [6]:
def lucas_solution(nums, queries):
    n = len(nums)
    best = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            s = lucas_score(nums[i:j + 1])
            best[i][j] = max(s, best[i][j - 1] if j > i else 0, best[i + 1][j] if i + 1 <= j else 0)
    return [best[l][r] for l, r in queries]

for nums, qs, exp in examples:
    print(lucas_solution(nums, qs), '| kỳ vọng', exp)

[12, 60, 60] | kỳ vọng [12, 60, 60]
[7, 14, 11, 14, 5] | kỳ vọng [7, 14, 11, 14, 5]


### 7.2. Cách 3 — DP khoảng `score` + `best` (lời giải chính)

In [7]:
def interval_dp(nums, queries):
    n = len(nums)
    best = [None] * n
    nxt_score = None
    nxt_best = None
    for i in range(n - 1, -1, -1):
        sc = [0] * n; bs = [0] * n
        sc[i] = bs[i] = nums[i]
        for j in range(i + 1, n):
            sc[j] = sc[j - 1] ^ nxt_score[j]
            b = sc[j]
            if bs[j - 1] > b:
                b = bs[j - 1]
            if nxt_best[j] > b:
                b = nxt_best[j]
            bs[j] = b
        best[i] = bs
        nxt_score, nxt_best = sc, bs
    return [best[l][r] for l, r in queries]

for nums, qs, exp in examples:
    print(interval_dp(nums, qs), '| kỳ vọng', exp)

[12, 60, 60] | kỳ vọng [12, 60, 60]
[7, 14, 11, 14, 5] | kỳ vọng [7, 14, 11, 14, 5]


### 7.3. Cách 4 — Offline theo đầu phải

Sắp truy vấn theo `r`. Khi thêm cột `r`, tính `score(i, r)` cho mọi `i` từ cột trước (`score(i, r) = score(i, r−1) ⊕ score(i+1, r)`), cập nhật `bestL[i] = max_{i ≤ i' ≤ j ≤ r} score(i', j)` bằng quét `i` từ phải sang trái. Chỉ giữ `O(n)` bộ nhớ — hữu ích khi `n` lớn hơn nhưng truy vấn ít.

In [8]:
def offline_by_r(nums, queries):
    n = len(nums)
    order = sorted(range(len(queries)), key=lambda t: queries[t][1])
    ans = [0] * len(queries)
    col = []                                   # col[i] = score(i, r)
    bestL = [0] * n                            # bestL[i] = max score của đoạn trong [i, r]
    qi = 0
    for r in range(n):
        new = [0] * (r + 1)
        new[r] = nums[r]
        for i in range(r - 1, -1, -1):
            new[i] = col[i] ^ new[i + 1]
        col = new
        run = 0
        for i in range(r, -1, -1):            # max(score(i..r, r)) theo hậu tố i, gộp với bestL cũ
            run = max(run, col[i])
            bestL[i] = max(bestL[i], run)
        while qi < len(order) and queries[order[qi]][1] == r:
            l = queries[order[qi]][0]
            ans[order[qi]] = bestL[l]
            qi += 1
    return ans

for nums, qs, exp in examples:
    print(offline_by_r(nums, qs), '| kỳ vọng', exp)

[12, 60, 60] | kỳ vọng [12, 60, 60]
[7, 14, 11, 14, 5] | kỳ vọng [7, 14, 11, 14, 5]


`bestL[i]` sau bước `r` = max của: các đoạn `[i', j]` với `j < r` (giá trị cũ, đã bao gồm `i' ≥ i`) và các đoạn kết thúc tại `r` bắt đầu từ `i' ≥ i` (biến `run`).

## 8. TRACE từng bước

In bảng `score` và `best` cho ví dụ 1 (4 phần tử đầu).

In [9]:
def trace(nums):
    n = len(nums)
    S = [[None] * n for _ in range(n)]; B = [[None] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        S[i][i] = B[i][i] = nums[i]
        for j in range(i + 1, n):
            S[i][j] = S[i][j - 1] ^ S[i + 1][j]
            B[i][j] = max(S[i][j], B[i][j - 1], B[i + 1][j])
    print('score[i][j] (hàng i):')
    for i in range(n):
        print('  ', ['.' if x is None else x for x in S[i]])
    print('best[i][j]:')
    for i in range(n):
        print('  ', ['.' if x is None else x for x in B[i]])

trace([2, 8, 4, 32])

score[i][j] (hàng i):
   [2, 10, 6, 46]
   ['.', 8, 12, 40]
   ['.', '.', 4, 36]
   ['.', '.', '.', 32]
best[i][j]:
   [2, 10, 12, 46]
   ['.', 8, 12, 40]
   ['.', '.', 4, 36]
   ['.', '.', '.', 32]


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maximumSubarrayXor(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        n = len(nums)
        best = [None] * n
        prev_sc = prev_bs = None
        for i in range(n - 1, -1, -1):
            sc = [0] * n
            bs = [0] * n
            sc[i] = bs[i] = nums[i]
            for j in range(i + 1, n):
                s = sc[j - 1] ^ prev_sc[j]
                sc[j] = s
                b = bs[j - 1]
                if s > b:
                    b = s
                pb = prev_bs[j]
                if pb > b:
                    b = pb
                bs[j] = b
            best[i] = bs
            prev_sc, prev_bs = sc, bs
        return [best[l][r] for l, r in queries]

for nums, qs, exp in examples:
    got = Solution().maximumSubarrayXor(nums, qs)
    print(got, 'OK' if got == exp else 'SAI')

[12, 60, 60] OK
[7, 14, 11, 14, 5] OK


## 10. Vì sao thuật toán đúng

1. **Truy hồi score:** gọi `T_t(i, j)` là phần tử đầu sau `t` bước của `a[i..j]`. Bằng quy nạp theo độ dài, score là tổ hợp XOR tuyến tính của các phần tử với hệ số `C(m−1, k) mod 2`; đẳng thức Pascal `C(m−1, k) = C(m−2, k−1) + C(m−2, k)` mod 2 cho đúng `score(i, j) = score(i, j−1) ⊕ score(i+1, j)` (kiểm chứng ở 3.2).
2. **Truy hồi best:** mọi đoạn con của `[i, j]` hoặc là chính `[i, j]`, hoặc nằm trong `[i, j−1]`, hoặc nằm trong `[i+1, j]` ⇒ max của ba giá trị.
3. **Thứ tự tính:** hàng `i` dùng hàng `i + 1` (đã tính) và ô bên trái cùng hàng.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(q n⁴)` | `O(n)` |
| 2. Lucas từng đoạn | `O(n³ + q)` | `O(n²)` |
| 3. DP khoảng | `O(n² + q)` | `O(n²)` |
| 4. Offline theo `r` | `O(n² + q log q)` | `O(n + q)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'lucas': lucas_solution, 'interval_dp': interval_dp, 'offline': offline_by_r,
         'submit': lambda a, q: Solution().maximumSubarrayXor(a, q)}
edge = [([5], [[0, 0]], [5]), ([0, 0], [[0, 1]], [0]), ([1, 1, 1], [[0, 2], [1, 1]], [1, 1]),
        ([3, 5], [[0, 1]], [6]), ([1, 2, 4, 8], [[0, 3]], [15])]
for nums, qs, exp in examples + edge:
    assert brute(nums, qs) == exp, (nums, qs, brute(nums, qs))
    for name, f in impls.items():
        assert f(nums, qs) == exp, (name, nums, qs)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng

In [12]:
rng = random.Random(3277)
cnt = 0
for _ in range(300):
    n = rng.randint(1, 9)
    nums = [rng.randint(0, 63) for _ in range(n)]
    qs = []
    for _ in range(rng.randint(1, 6)):
        l = rng.randrange(n); r = rng.randrange(l, n); qs.append([l, r])
    exp = brute(nums, qs)
    for name, f in impls.items():
        assert f(nums, qs) == exp, (name, nums, qs)
    cnt += 1
print('Khớp mô phỏng trên', cnt, 'bộ test.')

Khớp mô phỏng trên 300 bộ test.


### 12.3. Ca lớn `n = 2000`, `q = 10^5`

In [13]:
import time
N = 2000
nums = [rng.randint(0, 2 ** 31 - 1) for _ in range(N)]
qs = []
for _ in range(10 ** 5):
    l = rng.randrange(N); r = rng.randrange(l, N); qs.append([l, r])
t = time.time(); a = Solution().maximumSubarrayXor(nums, qs); t1 = time.time() - t
print(f'n = 2000, q = 1e5: nộp {t1:.2f}s')
small_nums = nums[:300]
small_qs = []
for _ in range(2000):
    l = rng.randrange(300); r = rng.randrange(l, 300); small_qs.append([l, r])
t = time.time(); b = offline_by_r(small_nums, small_qs); t2 = time.time() - t
assert b == Solution().maximumSubarrayXor(small_nums, small_qs)
print(f'offline (n = 300) khớp lời giải nộp | {t2:.2f}s')

n = 2000, q = 1e5: nộp 0.43s
offline (n = 300) khớp lời giải nộp | 0.01s


Kiểm tra độc lập: đáp án truy vấn `[l, r]` luôn `≥ max(nums[l..r])` (đoạn một phần tử) và không giảm khi mở rộng đoạn.

In [14]:
for (l, r), v in zip(qs[:2000], a[:2000]):
    assert v >= max(nums[l:r + 1])
best_tab = Solution().maximumSubarrayXor(nums[:200], [[0, r] for r in range(200)])
assert all(x <= y for x, y in zip(best_tab, best_tab[1:]))
print('đáp án >= max phần tử và không giảm theo r: đúng.')

đáp án >= max phần tử và không giảm theo r: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nghĩ score là XOR của **mọi** phần tử

In [15]:
def wrong_plain_xor(nums, queries):
    from functools import reduce
    ans = []
    for l, r in queries:
        ans.append(max(reduce(lambda u, v: u ^ v, nums[i:j + 1]) for i in range(l, r + 1) for j in range(i, r + 1)))
    return ans

print('sai :', wrong_plain_xor(*examples[0][:2]), '| đúng:', Solution().maximumSubarrayXor(*examples[0][:2]))

sai : [14, 60, 63] | đúng: [12, 60, 60]


**Vì sao sai:** hệ số là `C(m−1, k) mod 2`, không phải luôn 1. Ví dụ `[2, 8, 4]` có score `2 ⊕ 4 = 6` (phần tử giữa bị triệt tiêu).

### Lỗi 2 — Chỉ lấy score của **toàn đoạn** truy vấn

In [16]:
def wrong_whole_only(nums, queries):
    return [xor_score(nums[l:r + 1]) for l, r in queries]

print('sai :', wrong_whole_only(*examples[0][:2]), '| đúng:', Solution().maximumSubarrayXor(*examples[0][:2]))

sai : [6, 60, 27] | đúng: [12, 60, 60]


**Vì sao sai:** đề hỏi max trên **mọi đoạn con** của `nums[l..r]`.

### Lỗi 3 — Truy hồi kiểu XOR tiền tố `score(i, j) = score(i, j−1) ⊕ nums[j]`

In [17]:
def wrong_prefix_recurrence(nums, queries):
    n = len(nums)
    S = [[0] * n for _ in range(n)]; B = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        S[i][i] = B[i][i] = nums[i]
        for j in range(i + 1, n):
            S[i][j] = S[i][j - 1] ^ nums[j]                       # SAI
            B[i][j] = max(S[i][j], B[i][j - 1], B[i + 1][j])
    return [B[l][r] for l, r in queries]

print('sai :', wrong_prefix_recurrence(*examples[1][:2]), '| đúng:', Solution().maximumSubarrayXor(*examples[1][:2]))

sai : [7, 15, 10, 15, 5] | đúng: [7, 14, 11, 14, 5]


**Vì sao sai:** đó là XOR thường (lỗi 1 dưới dạng DP); truy hồi đúng ghép **hai đoạn chồng nhau** `[i, j−1]` và `[i+1, j]`.

### Lỗi 4 — Bảng `best` quên nhánh `best[i+1][j]`

In [18]:
def wrong_best_two_terms(nums, queries):
    n = len(nums)
    S = [[0] * n for _ in range(n)]; B = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        S[i][i] = B[i][i] = nums[i]
        for j in range(i + 1, n):
            S[i][j] = S[i][j - 1] ^ S[i + 1][j]
            B[i][j] = max(S[i][j], B[i][j - 1])                   # SAI: bỏ sót đoạn không chứa i
    return [B[l][r] for l, r in queries]

print('sai :', wrong_best_two_terms(*examples[0][:2]), '| đúng:', Solution().maximumSubarrayXor(*examples[0][:2]))

sai : [10, 60, 46] | đúng: [12, 60, 60]


**Vì sao sai:** bảng chỉ còn xét các đoạn **bắt đầu tại `i`** (hoặc kết thúc sớm hơn) — bỏ qua `[8, 4]` khi hỏi `[0, 2]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Phép "thay `a[i]` bằng `a[i] op a[i+1]` rồi bỏ phần tử cuối" ⇒ tam giác Pascal; với XOR là Pascal mod 2.
- "Max trên mọi đoạn con của `[l, r]`" với `n ≤ 2000` ⇒ bảng `best[i][j]` `O(n²)` trả lời `O(1)`.
- Tính hàng `i` từ hàng `i + 1` ⇒ có thể giữ ít hàng cho `score`.

## 15. Biến thể của bài

1. Phép cộng thay XOR ⇒ `score = Σ C(m−1, k) a[k]` (mod `p`) — vẫn truy hồi `score(i, j−1) + score(i+1, j)`.
2. Hỏi score **nhỏ nhất** ⇒ `min`.
3. Đếm số đoạn con có score bằng `0`.
4. LC 2221 (Find Triangular Sum of an Array) — phép cộng mod 10.

In [19]:
# Biến thể 1 (cộng mod p) và 4 (LC 2221)
def additive_score_table(nums, mod):
    n = len(nums)
    S = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        S[i][i] = nums[i] % mod
        for j in range(i + 1, n):
            S[i][j] = (S[i][j - 1] + S[i + 1][j]) % mod
    return S

def triangular_sum(nums):
    a = list(nums)
    while len(a) > 1:
        a = [(a[i] + a[i + 1]) % 10 for i in range(len(a) - 1)]
    return a[0]

from math import comb
rng = random.Random(15)
for _ in range(200):
    nums = [rng.randint(0, 9) for _ in range(rng.randint(1, 10))]
    S = additive_score_table(nums, 10)
    assert S[0][len(nums) - 1] == triangular_sum(nums) == sum(comb(len(nums) - 1, k) * v for k, v in enumerate(nums)) % 10
print('LC 2221: truy hồi khoảng = mô phỏng = công thức nhị thức. Ví dụ [1,2,3,4,5] ->', triangular_sum([1, 2, 3, 4, 5]))

LC 2221: truy hồi khoảng = mô phỏng = công thức nhị thức. Ví dụ [1,2,3,4,5] -> 8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mô phỏng | `O(q n⁴)` | Oracle |
| 2 | Lucas từng đoạn | `O(n³)` | Công thức đóng |
| 3 | DP khoảng | `O(n² + q)` | **Nộp bài** |
| 4 | Offline theo `r` | `O(n² + q log q)` | Bộ nhớ `O(n)` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 1]`, truy vấn `[0, 1]`.

**Bài 2 (tính tay):** score của `[a, b, c, d]` theo `a, b, c, d`.

**Bài 3 (điền TODO):** trả về **đoạn** đạt max cho mỗi truy vấn.

**Bài 4 (tự viết lại):** đếm số đoạn con của cả mảng có score bằng 0.

**Bài 5 (mini challenge):** với độ dài `m = 2^t`, score chỉ phụ thuộc phần tử nào? Chứng minh bằng Lucas và kiểm bằng code.

In [20]:
def argmax_segment_todo(nums, queries):
    # TODO
    raise NotImplementedError

def count_zero_score_todo(nums):
    # TODO
    raise NotImplementedError

def power_of_two_length_todo(a):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[1]` score 1, `[1, 1]` score 0. (2) Max. (3) `1`.

**Bài 2.** (1) `m − 1 = 3 = 11₂`. (2) Mọi `k ∈ {0,1,2,3}` là tập con của `11₂`. (3) `a ⊕ b ⊕ c ⊕ d`.

**Bài 3.** (1) Lưu kèm `(i, j)` trong bảng `best`. (2) Cập nhật cùng lúc với max. (3) Kiểm `xor_score` của đoạn trả về.

**Bài 4.** (1) Đếm số ô `score[i][j] == 0`. (2) `O(n²)`. (3) Vét cạn nhỏ.

**Bài 5.** (1) `m − 1 = 2^t − 1` có mọi bit thấp bằng 1. (2) Mọi `k < m` đều là tập con ⇒ score = XOR **mọi** phần tử. (3) Ngược lại, `m = 2^t + 1` ⇒ `m − 1 = 2^t` ⇒ chỉ phần tử đầu và cuối.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 5
from functools import reduce
assert Solution().maximumSubarrayXor([1, 1], [[0, 1]]) == [1]
for _ in range(100):
    a = [rng.randint(0, 50) for _ in range(4)]
    assert xor_score(a) == a[0] ^ a[1] ^ a[2] ^ a[3]
for t in range(1, 5):
    for _ in range(30):
        a = [rng.randint(0, 99) for _ in range(2 ** t)]
        assert xor_score(a) == reduce(lambda u, v: u ^ v, a)
        b = [rng.randint(0, 99) for _ in range(2 ** t + 1)]
        assert xor_score(b) == b[0] ^ b[-1]
print('Bài 1: [1] | Bài 2: a^b^c^d | Bài 5: m = 2^t -> XOR mọi phần tử; m = 2^t + 1 -> đầu ^ cuối.')

Bài 1: [1] | Bài 2: a^b^c^d | Bài 5: m = 2^t -> XOR mọi phần tử; m = 2^t + 1 -> đầu ^ cuối.


In [22]:
# Bài 3, 4
def argmax_segment(nums, queries):
    n = len(nums)
    S = [[0] * n for _ in range(n)]; B = [[None] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        S[i][i] = nums[i]; B[i][i] = (nums[i], i, i)
        for j in range(i + 1, n):
            S[i][j] = S[i][j - 1] ^ S[i + 1][j]
            B[i][j] = max((S[i][j], i, j), B[i][j - 1], B[i + 1][j])
    return [B[l][r] for l, r in queries]

def count_zero_score(nums):
    n = len(nums); cnt = 0
    prev = None
    for i in range(n - 1, -1, -1):
        sc = [0] * n; sc[i] = nums[i]; cnt += nums[i] == 0
        for j in range(i + 1, n):
            sc[j] = sc[j - 1] ^ prev[j]; cnt += sc[j] == 0
        prev = sc
    return cnt

for _ in range(200):
    n = rng.randint(1, 8); nums = [rng.randint(0, 7) for _ in range(n)]
    l = rng.randrange(n); r = rng.randrange(l, n)
    v, i, j = argmax_segment(nums, [[l, r]])[0]
    assert l <= i <= j <= r and xor_score(nums[i:j + 1]) == v == brute(nums, [[l, r]])[0]
    assert count_zero_score(nums) == sum(1 for i in range(n) for j in range(i, n) if xor_score(nums[i:j + 1]) == 0)
print('Bài 3, 4 khớp mô phỏng. Ví dụ đoạn tốt nhất của truy vấn [1,4] ví dụ 1:', argmax_segment(*examples[0][:2])[1])

Bài 3, 4 khớp mô phỏng. Ví dụ đoạn tốt nhất của truy vấn [1,4] ví dụ 1: (60, 1, 4)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for i = n-1..0:
    sc[i] = bs[i] = nums[i]
    for j = i+1..n-1:
        sc[j] = sc[j-1] ^ prev_sc[j]
        bs[j] = max(sc[j], bs[j-1], prev_bs[j])
    best[i] = bs
answer = best[l][r]
```

**Checklist:**

- [ ] Truy hồi `score(i, j−1) ⊕ score(i+1, j)` (không phải XOR thường)?
- [ ] `best` gộp đủ ba nhánh?
- [ ] Duyệt `i` giảm dần?
- [ ] Bộ nhớ `O(n²)` cho `best` (n ≤ 2000 ổn)?
- [ ] Truy vấn trả lời `O(1)`?